In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim

#Task 1

df = pd.read_csv("heart.csv")

y = df["target"].values
X = df.drop(columns=["target"]).values

print("Баланс класів:")
print(pd.Series(y).value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

#Task 2

input_size = X_train.shape[1]

model = nn.Sequential(
    nn.Linear(input_size, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

print(model)

#Task 3

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30

losses = []
accuracies = []

for epoch in range(epochs):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    preds = (outputs >= 0.5).float()
    acc = (preds == y_train).float().mean()

    losses.append(loss.item())
    accuracies.append(acc.item())

    print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Accuracy={acc.item():.4f}")

with torch.no_grad():
    test_outputs = model(X_test)
    test_preds = (test_outputs >= 0.5).float()

    test_acc = (test_preds == y_test).float().mean()
    test_loss = criterion(test_outputs, y_test)

print("\nФінальні метрики:")
print("Loss:", test_loss.item())
print("Accuracy:", test_acc.item())

#Task 4

plt.figure()
plt.plot(losses, label="Loss")
plt.xlabel("Епоха")
plt.ylabel("Loss")
plt.title("Loss під час навчання")
plt.legend()
plt.savefig("loss_healthrisk_mlp.png")
plt.show()

plt.figure()
plt.plot(accuracies, label="Accuracy")
plt.xlabel("Епоха")
plt.ylabel("Accuracy")
plt.title("Accuracy під час навчання")
plt.legend()
plt.savefig("accuracy_healthrisk_mlp.png")
plt.show()


print("\nВисновок:")
print("Якщо Loss зменшується, а Accuracy зростає — модель навчається добре.")